# Moving a picture to a coarser one

You have a fine picture — integers, a high-resolution grid, a source program. You move it to a coarser picture — even or odd, a low-resolution grid, a compiled binary. The honest way to do that is a *commuting square*: look after the move, or move after the look. The two routes agree.

That agreement is the transport certificate $\chi_F$. It says the move is compatible with looking. It does not, by itself, decide anything in the coarser world.

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_mur_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "mur" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "mur"
elif (p / "mur" / "code").is_dir():
    root = p / "mur"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
repo_root = root.parents[2] if len(root.parents) >= 3 else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from sympy import Rational
from coda_mur_core import (
    DescentSquareCertificate,
    LossLedger,
    MorphismKind,
    TerminalRegime,
    evaluate_transport_authority,
)

## The square, in words

Fine states are integers. Coarse states are bits: $0$ for even, $1$ for odd. The move of states is $F(n)=n \bmod 2$. Looking after $F$ should match moving the look. When it does, the calculator records a commuting certificate.

An *exact* move (nothing forgotten) needs no loss ledger. A *quotient* (many integers become one bit) does.

In [2]:
exact = DescentSquareCertificate(
    source_regime="integers",
    target_regime="integers_again",
    morphism_name="identity",
    morphism_kind=MorphismKind.EXACT,
    chi_certified=True,
    commutes=True,
    carrier=("2", "3", "4"),
    state_map=lambda x: x,
    observation_map=lambda x: x,
    source_look=lambda x: x,
    target_look=lambda x: x,
)
print("exact square valid?", exact.is_valid())
assert exact.is_valid() is True

target = TerminalRegime(
    regime_id="integers_again",
    terminalize=lambda y: f"term_{y}",
    is_auth=lambda t: True,
)
stood = evaluate_transport_authority("4", target, "4", exact)
print("inhabited:", stood.inhabited, "terminal:", stood.target_terminal)
assert stood.inhabited is True
assert stood.refusal_code is None

exact square valid? True
inhabited: True terminal: term_4


## A coarser picture: integers to parity

Now the move really forgets magnitude. $2$ and $4$ become the same bit. That is a quotient. The square can still commute — looking at the bit of $n$ is the same as taking parity of the recorded integer — but the calculator requires a non-empty loss ledger that says so.

In [3]:
parity = DescentSquareCertificate(
    source_regime="integers",
    target_regime="parity",
    morphism_name="mod2",
    morphism_kind=MorphismKind.QUOTIENT,
    chi_certified=True,
    commutes=True,
    loss_ledger=LossLedger(e=True, loss_mass=Rational(1, 1)),
    carrier=("2", "3", "4"),
    state_map=lambda x: str(int(x) % 2),
    observation_map=lambda x: str(int(x) % 2),
    source_look=lambda x: x,
    target_look=lambda x: x,
)
print("quotient square valid?", parity.is_valid())
assert parity.is_valid() is True

parity_target = TerminalRegime(
    regime_id="parity",
    terminalize=lambda y: f"bit_{y}",
    is_auth=lambda t: True,
)
even_four = evaluate_transport_authority("4", parity_target, "0", parity)
even_two = evaluate_transport_authority("2", parity_target, "0", parity)
print("4 and 2 both land on bit 0:", even_four.target_terminal, even_two.target_terminal)
assert even_four.inhabited is True
assert even_two.inhabited is True
assert even_four.target_terminal == even_two.target_terminal

quotient square valid? True
4 and 2 both land on bit 0: bit_0 bit_0


## When the square does not commute

Declare the same move, but say the square fails. Looking after the move no longer matches moving after the look. The calculator refuses with `R_notransport`. A broken square is not a coarser picture; it is a refusal to transport.

In [4]:
broken = DescentSquareCertificate(
    source_regime="integers",
    target_regime="parity",
    morphism_name="mod2_broken",
    morphism_kind=MorphismKind.QUOTIENT,
    chi_certified=True,
    commutes=False,
    loss_ledger=LossLedger(e=True, loss_mass=Rational(1, 1)),
    carrier=("2", "3", "4"),
    state_map=lambda x: str(int(x) % 2),
    observation_map=lambda x: str(int(x) % 2),
    source_look=lambda x: x,
    target_look=lambda x: x,
)
refused = evaluate_transport_authority("4", parity_target, "0", broken)
print("inhabited:", refused.inhabited, "refusal:", refused.refusal_code)
assert refused.inhabited is False
assert refused.refusal_code == "R_notransport"

missing = evaluate_transport_authority("4", parity_target, "0", None)
print("no certificate:", missing.inhabited, missing.refusal_code)
assert missing.refusal_code == "R_notransport"

inhabited: False refusal: R_notransport
no certificate: False R_notransport
